In [3]:
import pandas as pd

In [4]:
df = pd.read_parquet("processed_reviews.parquet")

In [5]:
df.columns

Index(['recommendationid', 'appid', 'game', 'author_steamid',
       'author_num_games_owned', 'author_num_reviews',
       'author_playtime_forever', 'author_playtime_last_two_weeks',
       'author_playtime_at_review', 'author_last_played', 'language', 'review',
       'timestamp_created', 'timestamp_updated', 'voted_up', 'votes_up',
       'votes_funny', 'weighted_vote_score', 'comment_count', 'steam_purchase',
       'received_for_free', 'written_during_early_access',
       'hidden_in_steam_china', 'steam_china_location'],
      dtype='str')

In [9]:
appids = df.appid.unique()
users= df.author_steamid.unique()

In [17]:
from pathlib import Path
import threading
import time

import duckdb
import pandas as pd


# ============================================================
# MAJOR PARAMETERS
# ============================================================

EDGE_OUTPUT_FILE = "game_edges.parquet"
GAME_LOOKUP_FILE = "game_lookup.parquet"
DATABASE_FILE = "recommendation_build.duckdb"
TEMP_DIRECTORY = "duckdb_temp"

USER_COLUMN = "author_steamid"
GAME_COLUMN = "appid"
GAME_NAME_COLUMN = "game"
VOTE_COLUMN = "voted_up"
TIMESTAMP_COLUMN = "timestamp_updated"

# Adjust these for your computer
MEMORY_LIMIT = "80GB"
THREADS = 16

# Keep all observed edges for now.
# Weak edges can be removed later without rebuilding.
MIN_SHARED_REVIEWS = 1

# Progress update frequency
PROGRESS_POLL_SECONDS = 2.0


# ============================================================
# PROJECT PATHS
# ============================================================

PROJECT_DIRECTORY = Path.cwd()

edge_output_path = (
    PROJECT_DIRECTORY / EDGE_OUTPUT_FILE
).as_posix()

game_lookup_path = (
    PROJECT_DIRECTORY / GAME_LOOKUP_FILE
).as_posix()

database_path = (
    PROJECT_DIRECTORY / DATABASE_FILE
).as_posix()

temp_path = (
    PROJECT_DIRECTORY / TEMP_DIRECTORY
).as_posix()

Path(temp_path).mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def directory_size_gb(directory: str) -> float:
    """
    Return the total size of a directory in GiB.
    """

    directory_path = Path(directory)

    if not directory_path.exists():
        return 0.0

    total_bytes = sum(
        file_path.stat().st_size
        for file_path in directory_path.rglob("*")
        if file_path.is_file()
    )

    return total_bytes / 1024**3


def run_query_with_progress(
    connection: duckdb.DuckDBPyConnection,
    sql: str,
    description: str,
    temp_directory: str,
    poll_seconds: float = 2.0,
) -> None:
    """
    Run a DuckDB query in a background thread while printing
    progress information in the main thread.

    This is useful in VS Code, where DuckDB's native terminal
    progress bar may not display.
    """

    result = {
        "error": None,
    }

    def execute_query() -> None:
        try:
            connection.execute(sql)

        except Exception as error:
            result["error"] = error

    worker = threading.Thread(
        target=execute_query,
        daemon=True,
    )

    start_time = time.perf_counter()
    worker.start()

    last_print_time = 0.0

    while worker.is_alive():
        current_time = time.perf_counter()

        if current_time - last_print_time >= poll_seconds:
            elapsed_seconds = current_time - start_time
            elapsed_minutes = elapsed_seconds / 60

            try:
                progress = connection.query_progress()
            except Exception:
                progress = None

            temp_size = directory_size_gb(temp_directory)

            if progress is not None and progress >= 0:
                # DuckDB versions may expose either 0–1 or 0–100.
                if progress <= 1:
                    progress_percent = progress * 100
                else:
                    progress_percent = progress

                progress_text = f"{progress_percent:5.1f}%"

            else:
                progress_text = "working"

            print(
                f"{description}: "
                f"{progress_text} | "
                f"{elapsed_minutes:,.1f} min elapsed | "
                f"temp disk: {temp_size:,.2f} GiB",
                flush=True,
            )

            last_print_time = current_time

        time.sleep(0.25)

    worker.join()

    elapsed_minutes = (
        time.perf_counter() - start_time
    ) / 60

    if result["error"] is not None:
        print(
            f"{description}: FAILED after "
            f"{elapsed_minutes:,.1f} minutes."
        )

        raise result["error"]

    print(
        f"{description}: complete in "
        f"{elapsed_minutes:,.1f} minutes."
    )


# ============================================================
# VALIDATE THE EXISTING DATAFRAME
# ============================================================

required_columns = {
    USER_COLUMN,
    GAME_COLUMN,
    GAME_NAME_COLUMN,
    VOTE_COLUMN,
    TIMESTAMP_COLUMN,
}

missing_columns = required_columns.difference(df.columns)

if missing_columns:
    raise ValueError(
        f"Missing required columns: {sorted(missing_columns)}"
    )

print("Input DataFrame validated.")
print(f"Input rows: {len(df):,}")


# ============================================================
# CONNECT TO DUCKDB
# ============================================================

con = duckdb.connect(database_path)

try:
    con.execute(
        f"SET memory_limit = '{MEMORY_LIMIT}'"
    )

    con.execute(
        f"SET threads = {THREADS}"
    )

    con.execute(
        f"SET temp_directory = '{temp_path}'"
    )

    # The custom progress function is used instead.
    con.execute(
        "SET enable_progress_bar = false"
    )

    # Make the existing pandas DataFrame available in DuckDB.
    con.register(
        "source_df",
        df,
    )

    print()
    print("DuckDB connected.")
    print(f"Memory limit: {MEMORY_LIMIT}")
    print(f"Threads: {THREADS}")
    print(f"Temporary directory: {temp_path}")


    # ========================================================
    # STAGE 1: CREATE LEAN REVIEW TABLE
    # ========================================================
    #
    # This removes the huge review text and all unrelated
    # columns before generating game pairs.
    #
    # If the same user reviewed the same app more than once,
    # retain the newest review.
    # ========================================================

    print()
    print("Stage 1/3: Creating lean review table...")

    stage_start = time.perf_counter()

    con.execute("DROP TABLE IF EXISTS reviews")

    con.execute(f"""
        CREATE TABLE reviews AS

        SELECT
            CAST("{USER_COLUMN}" AS UBIGINT)
                AS author_steamid,

            CAST("{GAME_COLUMN}" AS UINTEGER)
                AS appid,

            CAST("{VOTE_COLUMN}" AS BOOLEAN)
                AS voted_up

        FROM source_df

        WHERE "{USER_COLUMN}" IS NOT NULL
          AND "{GAME_COLUMN}" IS NOT NULL
          AND "{VOTE_COLUMN}" IS NOT NULL

        QUALIFY ROW_NUMBER() OVER (
            PARTITION BY
                "{USER_COLUMN}",
                "{GAME_COLUMN}"

            ORDER BY
                "{TIMESTAMP_COLUMN}" DESC
        ) = 1
    """)

    stage_elapsed = (
        time.perf_counter() - stage_start
    ) / 60

    lean_review_count = con.execute("""
        SELECT COUNT(*)
        FROM reviews
    """).fetchone()[0]

    print(
        f"Stage 1 complete in "
        f"{stage_elapsed:,.1f} minutes."
    )

    print(
        f"Lean review rows: "
        f"{lean_review_count:,}"
    )


    # ========================================================
    # STAGE 2: BUILD APPID → GAME LOOKUP
    # ========================================================
    #
    # If one appid has multiple observed names, choose the most
    # frequently occurring name.
    # ========================================================

    print()
    print("Stage 2/3: Building game-name lookup...")

    stage_start = time.perf_counter()

    con.execute(f"""
        COPY (
            WITH name_counts AS (
                SELECT
                    CAST("{GAME_COLUMN}" AS UINTEGER)
                        AS appid,

                    CAST("{GAME_NAME_COLUMN}" AS VARCHAR)
                        AS game,

                    COUNT(*) AS name_frequency

                FROM source_df

                WHERE "{GAME_COLUMN}" IS NOT NULL
                  AND "{GAME_NAME_COLUMN}" IS NOT NULL

                GROUP BY
                    "{GAME_COLUMN}",
                    "{GAME_NAME_COLUMN}"
            ),

            ranked_names AS (
                SELECT
                    appid,
                    game,
                    name_frequency,

                    ROW_NUMBER() OVER (
                        PARTITION BY appid

                        ORDER BY
                            name_frequency DESC,
                            game
                    ) AS name_rank

                FROM name_counts
            )

            SELECT
                appid,
                game

            FROM ranked_names

            WHERE name_rank = 1
        )

        TO '{game_lookup_path}'
        (
            FORMAT PARQUET,
            COMPRESSION ZSTD
        )
    """)

    stage_elapsed = (
        time.perf_counter() - stage_start
    ) / 60

    print(
        f"Stage 2 complete in "
        f"{stage_elapsed:,.1f} minutes."
    )

    print(
        f"Created: {GAME_LOOKUP_FILE}"
    )


    # ========================================================
    # STAGE 3: BUILD SPARSE GAME EDGES
    # ========================================================
    #
    # a.appid < b.appid guarantees:
    #
    # 1. No game is paired with itself.
    # 2. A-B and B-A are not both created.
    # 3. appid_a is always smaller than appid_b.
    #
    # Every user who reviewed both games increments:
    #
    # shared_review_count += 1
    #
    # Users who liked both also increment:
    #
    # both_positive_count += 1
    # ========================================================

    edge_sql = f"""
        COPY (
            SELECT
                a.appid AS appid_a,
                b.appid AS appid_b,

                SUM(
                    CASE
                        WHEN a.voted_up
                         AND b.voted_up
                        THEN 1
                        ELSE 0
                    END
                )::UBIGINT
                    AS both_positive_count,

                COUNT(*)::UBIGINT
                    AS shared_review_count,

                SUM(
                    CASE
                        WHEN a.voted_up
                         AND b.voted_up
                        THEN 1
                        ELSE 0
                    END
                )::DOUBLE
                / COUNT(*)
                    AS raw_score

            FROM reviews AS a

            INNER JOIN reviews AS b
                ON a.author_steamid
                 = b.author_steamid

               AND a.appid
                 < b.appid

            GROUP BY
                a.appid,
                b.appid

            HAVING COUNT(*) >= {MIN_SHARED_REVIEWS}
        )

        TO '{edge_output_path}'
        (
            FORMAT PARQUET,
            COMPRESSION ZSTD
        )
    """

    print()
    print("Stage 3/3: Building sparse game edges...")
    print("This is the main long-running step.")

    run_query_with_progress(
        connection=con,
        sql=edge_sql,
        description="Building game edges",
        temp_directory=temp_path,
        poll_seconds=PROGRESS_POLL_SECONDS,
    )


    # ========================================================
    # FINAL SUMMARY
    # ========================================================

    print()
    print("Calculating final summary...")

    user_count = con.execute("""
        SELECT COUNT(DISTINCT author_steamid)
        FROM reviews
    """).fetchone()[0]

    game_count = con.execute("""
        SELECT COUNT(DISTINCT appid)
        FROM reviews
    """).fetchone()[0]

    edge_count = con.execute(f"""
        SELECT COUNT(*)
        FROM read_parquet('{edge_output_path}')
    """).fetchone()[0]

    edge_file_size = (
        Path(edge_output_path).stat().st_size
        / 1024**3
    )

    lookup_file_size = (
        Path(game_lookup_path).stat().st_size
        / 1024**2
    )

    print()
    print("=" * 60)
    print("BUILD COMPLETE")
    print("=" * 60)

    print(
        f"Reviews used:       "
        f"{lean_review_count:,}"
    )

    print(
        f"Users:              "
        f"{user_count:,}"
    )

    print(
        f"Games:              "
        f"{game_count:,}"
    )

    print(
        f"Unique game edges:  "
        f"{edge_count:,}"
    )

    print(
        f"Edge file size:     "
        f"{edge_file_size:,.2f} GiB"
    )

    print(
        f"Lookup file size:   "
        f"{lookup_file_size:,.2f} MiB"
    )

    print()
    print(
        f"Created: {EDGE_OUTPUT_FILE}"
    )

    print(
        f"Created: {GAME_LOOKUP_FILE}"
    )

finally:
    try:
        con.unregister("source_df")
    except Exception:
        pass

    con.close()

Input DataFrame validated.
Input rows: 79,844,333

DuckDB connected.
Memory limit: 80GB
Threads: 16
Temporary directory: d:/Python Work/Game Recommendations/all_reviews.csv/duckdb_temp

Stage 1/3: Creating lean review table...
Stage 1 complete in 1.5 minutes.
Lean review rows: 79,842,036

Stage 2/3: Building game-name lookup...
Stage 2 complete in 1.2 minutes.
Created: game_lookup.parquet

Stage 3/3: Building sparse game edges...
This is the main long-running step.
Building game edges: working | 0.0 min elapsed | temp disk: 0.00 GiB
Building game edges: working | 0.0 min elapsed | temp disk: 0.00 GiB
Building game edges: working | 0.1 min elapsed | temp disk: 0.00 GiB
Building game edges: working | 0.1 min elapsed | temp disk: 0.00 GiB
Building game edges: working | 0.1 min elapsed | temp disk: 0.00 GiB
Building game edges: working | 0.2 min elapsed | temp disk: 0.00 GiB
Building game edges: working | 0.2 min elapsed | temp disk: 0.00 GiB
Building game edges: working | 0.2 min elapsed 

In [43]:
from pathlib import Path
import duckdb


# ============================================================
# MAJOR PARAMETERS
# ============================================================

INPUT_EDGE_FILE = "game_edges.parquet"
OUTPUT_EDGE_FILE = "game_edges_scored.parquet"

PRIOR_MEAN = 0.50
PRIOR_STRENGTH = 100

MEMORY_LIMIT = "80GB"
THREADS = 16


# ============================================================
# PATHS
# ============================================================

PROJECT_DIR = Path.cwd()

input_path = (PROJECT_DIR / INPUT_EDGE_FILE).as_posix()
output_path = (PROJECT_DIR / OUTPUT_EDGE_FILE).as_posix()


# ============================================================
# CREATE SCORED EDGE FILE
# ============================================================

con = duckdb.connect()

con.execute(f"SET memory_limit = '{MEMORY_LIMIT}'")
con.execute(f"SET threads = {THREADS}")

print("Calculating Bayesian-smoothed scores...")

con.execute(f"""
    COPY (
        SELECT
            appid_a,
            appid_b,
            both_positive_count,
            shared_review_count,
            raw_score,

            (
                both_positive_count
                + {PRIOR_MEAN} * {PRIOR_STRENGTH}
            )
            /
            (
                shared_review_count
                + {PRIOR_STRENGTH}
            ) AS smoothed_score

        FROM read_parquet('{input_path}')
    )
    TO '{output_path}'
    (
        FORMAT PARQUET,
        COMPRESSION ZSTD
    )
""")

con.close()

print(f"Created: {OUTPUT_EDGE_FILE}")

Calculating Bayesian-smoothed scores...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Created: game_edges_scored.parquet


In [46]:
import duckdb
import pandas as pd


# ============================================================
# MAJOR PARAMETERS
# ============================================================

EDGE_FILE = "game_edges_scored.parquet"
LOOKUP_FILE = "game_lookup.parquet"

# Use either TARGET_APPID or GAME_SEARCH.
# When TARGET_APPID is not None, it takes priority.
TARGET_APPID = 292030
GAME_SEARCH = ""

MIN_SHARED_REVIEWS = 5
RESULT_LIMIT = 30


# ============================================================
# CONNECT
# ============================================================

con = duckdb.connect()


# ============================================================
# RESOLVE GAME NAME TO APPID
# ============================================================

if TARGET_APPID is None:
    matches = con.execute(
        f"""
        SELECT
            appid,
            game
        FROM read_parquet(?)
        WHERE game ILIKE ?
        ORDER BY
            CASE
                WHEN lower(game) = lower(?) THEN 0
                WHEN lower(game) LIKE lower(?) THEN 1
                ELSE 2
            END,
            length(game),
            game
        LIMIT 20
        """,
        [
            LOOKUP_FILE,
            f"%{GAME_SEARCH}%",
            GAME_SEARCH,
            f"{GAME_SEARCH}%",
        ],
    ).df()

    if matches.empty:
        con.close()
        raise ValueError(
            f"No game found matching: {GAME_SEARCH!r}"
        )

    print("Matching games:")
    display(matches)

    TARGET_APPID = int(matches.iloc[0]["appid"])
    target_game = matches.iloc[0]["game"]

else:
    target_match = con.execute(
        """
        SELECT game
        FROM read_parquet(?)
        WHERE appid = ?
        LIMIT 1
        """,
        [LOOKUP_FILE, TARGET_APPID],
    ).fetchone()

    target_game = (
        target_match[0]
        if target_match is not None
        else "Unknown game"
    )


print()
print(f"Recommendations for: {target_game}")
print(f"App ID: {TARGET_APPID:,}")


# ============================================================
# QUERY BOTH SIDES OF THE SYMMETRIC EDGE TABLE
# ============================================================

recommendations = con.execute(
    f"""
    WITH connections AS (
        SELECT
            CASE
                WHEN edges.appid_a = ?
                THEN edges.appid_b
                ELSE edges.appid_a
            END AS connected_appid,

            edges.both_positive_count,
            edges.shared_review_count,
            edges.raw_score,
            edges.smoothed_score

        FROM read_parquet(?) AS edges

        WHERE edges.appid_a = ?
           OR edges.appid_b = ?
    )

    SELECT
        connections.connected_appid AS appid,
        lookup.game,

        connections.both_positive_count,
        connections.shared_review_count,

        connections.raw_score,
        connections.smoothed_score

    FROM connections

    LEFT JOIN read_parquet(?) AS lookup
        ON connections.connected_appid = lookup.appid

    WHERE connections.shared_review_count >= ?

    ORDER BY
        connections.smoothed_score DESC,
        connections.shared_review_count DESC

    LIMIT ?
    """,
    [
        TARGET_APPID,
        EDGE_FILE,
        TARGET_APPID,
        TARGET_APPID,
        LOOKUP_FILE,
        MIN_SHARED_REVIEWS,
        RESULT_LIMIT,
    ],
).df()

con.close()

recommendations


Recommendations for: The Witcher 3: Wild Hunt
App ID: 292,030


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,appid,game,both_positive_count,shared_review_count,raw_score,smoothed_score
0,378648,The Witcher 3: Wild Hunt - Blood and Wine,6398,6591,0.970718,0.963683
1,378649,The Witcher 3: Wild Hunt - Hearts of Stone,4710,4905,0.960245,0.951049
2,355880,The Witcher 3: Wild Hunt - Expansion Pass,2824,2925,0.965470,0.950083
3,431960,Wallpaper Engine,37892,40147,0.943831,0.942729
4,620,Portal 2,20815,22125,0.940791,0.938808
5,1145360,Hades,19938,21213,0.939895,0.937831
6,413150,Stardew Valley,27388,29158,0.939296,0.937795
7,207610,The Walking Dead,6689,7105,0.941450,0.935323
8,227300,Euro Truck Simulator 2,23841,25463,0.936300,0.934593
9,250320,The Wolf Among Us,4346,4612,0.942324,0.932937


In [47]:
#final table creation

from pathlib import Path
import duckdb


# ============================================================
# PARAMETERS
# ============================================================

EDGE_FILE = "game_edges_scored.parquet"
OUTPUT_FILE = "game_recommendations.parquet"

TOP_N = 50
MIN_SHARED_REVIEWS = 5

MEMORY_LIMIT = "80GB"
THREADS = 16

PROJECT_DIR = Path.cwd()

edge_path = (PROJECT_DIR / EDGE_FILE).as_posix()
output_path = (PROJECT_DIR / OUTPUT_FILE).as_posix()


# ============================================================
# BUILD FINAL RECOMMENDATION TABLE
# ============================================================

con = duckdb.connect()

try:
    con.execute(f"SET memory_limit = '{MEMORY_LIMIT}'")
    con.execute(f"SET threads = {THREADS}")

    print("Expanding symmetric edges into directed recommendations...")

    con.execute(f"""
        COPY (
            WITH directed_edges AS (

                -- A recommends B
                SELECT
                    appid_a AS source_appid,
                    appid_b AS recommended_appid,
                    both_positive_count,
                    shared_review_count,
                    raw_score,
                    smoothed_score

                FROM read_parquet('{edge_path}')

                WHERE shared_review_count >= {MIN_SHARED_REVIEWS}


                UNION ALL


                -- B recommends A
                SELECT
                    appid_b AS source_appid,
                    appid_a AS recommended_appid,
                    both_positive_count,
                    shared_review_count,
                    raw_score,
                    smoothed_score

                FROM read_parquet('{edge_path}')

                WHERE shared_review_count >= {MIN_SHARED_REVIEWS}
            ),

            ranked AS (
                SELECT
                    source_appid,
                    recommended_appid,
                    both_positive_count,
                    shared_review_count,
                    raw_score,
                    smoothed_score,

                    ROW_NUMBER() OVER (
                        PARTITION BY source_appid

                        ORDER BY
                            smoothed_score DESC,
                            shared_review_count DESC,
                            recommended_appid
                    ) AS recommendation_rank

                FROM directed_edges
            )

            SELECT
                source_appid,
                recommended_appid,
                recommendation_rank,
                smoothed_score AS score,
                shared_review_count,
                both_positive_count,
                raw_score

            FROM ranked

            WHERE recommendation_rank <= {TOP_N}

            ORDER BY
                source_appid,
                recommendation_rank
        )

        TO '{output_path}'
        (
            FORMAT PARQUET,
            COMPRESSION ZSTD
        )
    """)

    row_count = con.execute(f"""
        SELECT COUNT(*)
        FROM read_parquet('{output_path}')
    """).fetchone()[0]

    game_count = con.execute(f"""
        SELECT COUNT(DISTINCT source_appid)
        FROM read_parquet('{output_path}')
    """).fetchone()[0]

    print("Recommendation table complete.")
    print(f"Rows:  {row_count:,}")
    print(f"Games: {game_count:,}")
    print(f"File:  {OUTPUT_FILE}")

finally:
    con.close()

Expanding symmetric edges into directed recommendations...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Recommendation table complete.
Rows:  2,112,697
Games: 59,745
File:  game_recommendations.parquet


In [50]:
import duckdb

TARGET_APPID = 433340
RESULT_LIMIT = 10

con = duckdb.connect()

recommendations = con.execute(
    """
    SELECT
        r.recommended_appid AS appid,
        l.game,
        r.recommendation_rank,
        r.score,
        r.shared_review_count,
        r.both_positive_count,
        r.raw_score

    FROM read_parquet('game_recommendations.parquet') AS r

    LEFT JOIN read_parquet('game_lookup.parquet') AS l
        ON r.recommended_appid = l.appid

    WHERE r.source_appid = ?

    ORDER BY r.recommendation_rank

    LIMIT ?
    """,
    [TARGET_APPID, RESULT_LIMIT],
).df()

con.close()

recommendations

,appid,game,recommendation_rank,score,shared_review_count,both_positive_count,raw_score
0,620,Portal 2,1,0.960801,8191,7916,0.966427
1,413150,Stardew Valley,2,0.959557,13252,12762,0.963024
2,105600,Terraria,3,0.956250,19237,18441,0.958621
3,960090,Bloons TD 6,4,0.954626,7217,6935,0.960926
4,1281930,tModLoader,5,0.954329,4936,4756,0.963533
5,431960,Wallpaper Engine,6,0.953551,6574,6314,0.960450
6,1118200,People Playground,7,0.953227,4176,4026,0.964080
7,264710,Subnautica,8,0.950301,7385,7063,0.956398
8,550,Left 4 Dead 2,9,0.948139,7208,6879,0.954356
9,3590,Plants vs. Zombies: Game of the Year,10,0.946467,3113,2991,0.960810
